In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. LOAD DATA
# ============================================================

metadata = pd.read_parquet("metadata.parquet")

features = ["gender", "serve_number", "height_cm"]
target = "speed_kmh"

data = metadata[features + [target, "server"]].copy()

# Convert numeric columns
for col in ["serve_number", "height_cm", "speed_kmh"]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Keep rows with valid target and player
data = data.dropna(subset=[target, "server"])

# Keep valid serve numbers and gender categories
data = data[
    data["serve_number"].isin([1, 2])
    & data["gender"].isin(["M", "F"])
].copy()

print("Number of serves:", len(data))
print("Number of players:", data["server"].nunique())


# ============================================================
# 2. SPLIT PLAYERS BY HEIGHT DISTRIBUTION
# ============================================================

# One row per player, with their median recorded height
players = (
    data.groupby("server", as_index=False)
    .agg(height_cm=("height_cm", "median"))
)

# Fill missing heights only to determine the split
median_height = players["height_cm"].median()
players["height_for_split"] = players["height_cm"].fillna(median_height)

# Divide players into height groups
players["height_group"] = pd.qcut(
    players["height_for_split"],
    q=4,
    labels=False,
    duplicates="drop"
)

# Stratified split of players, not individual serves
train_players, test_players = train_test_split(
    players["server"],
    test_size=0.2,
    random_state=42,
    stratify=players["height_group"]
)

# All serves from a player stay in the same set
train_data = data[data["server"].isin(train_players)].copy()
test_data = data[data["server"].isin(test_players)].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("\nDATA SPLIT")
print("Training serves:", len(train_data))
print("Test serves:", len(test_data))
print("Training players:", train_data["server"].nunique())
print("Test players:", test_data["server"].nunique())

print("\nHeights of test players:")
test_heights = (
    test_data.groupby("server")["height_cm"]
    .first()
    .sort_values()
)
print(test_heights.to_string())

print("\nDistinct heights in test set:", test_data["height_cm"].nunique())


# ============================================================
# 3. PREPROCESSING
# ============================================================

categorical_features = ["gender", "serve_number"]
numeric_features = ["height_cm"]

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_features),
    ("num", numeric_transformer, numeric_features)
])


# ============================================================
# 4. DEFINE AND TRAIN MODELS
# ============================================================

# Baseline: predict the mean speed in the training set
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

# Linear regression
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)


# ============================================================
# 5. EVALUATE MODEL PERFORMANCE
# ============================================================

models = {
    "Mean baseline": baseline,
    "Linear regression": model
}

results = []

for name, fitted_model in models.items():

    predictions = fitted_model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE (km/h)": mae,
        "RMSE (km/h)": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

print("\nMODEL PERFORMANCE")
print(results_df.round(3).to_string(index=False))


# ============================================================
# 6. EXAMPLE PREDICTIONS
# ============================================================

# Choose how many examples to display
n_examples = 30  # Change this number as desired

# Select at most one serve per player in the first pass,
# then add more serves from the same players if needed
rng = np.random.default_rng(42)

# Shuffle the test set
shuffled_indices = rng.permutation(X_test.index)

selected_indices = []
selected_players = set()

# First pass: prioritize different players
for idx in shuffled_indices:
    player = test_data.loc[idx, "server"]

    if player not in selected_players:
        selected_indices.append(idx)
        selected_players.add(player)

    if len(selected_indices) >= min(n_examples, len(X_test)):
        break

# Second pass: add more serves from players already selected
if len(selected_indices) < min(n_examples, len(X_test)):
    for idx in shuffled_indices:
        if idx not in selected_indices:
            selected_indices.append(idx)

        if len(selected_indices) >= min(n_examples, len(X_test)):
            break

# Build the examples table
examples = X_test.loc[selected_indices].copy()

examples.insert(
    0,
    "Player",
    test_data.loc[selected_indices, "server"]
)

examples["Actual speed (km/h)"] = y_test.loc[selected_indices]

examples["Predicted speed (km/h)"] = model.predict(
    X_test.loc[selected_indices]
)

examples["Error (km/h)"] = (
    examples["Actual speed (km/h)"]
    - examples["Predicted speed (km/h)"]
)

# Display results
print(f"\nEXAMPLE PREDICTIONS ({len(examples)} serves)")
print("Number of different players:", examples["Player"].nunique())
print(examples.round(2).to_string(index=False))


# ============================================================
# 7. PREDICTION ERRORS
# ============================================================

test_predictions = model.predict(X_test)
errors = y_test.to_numpy() - test_predictions

print("\nPREDICTION ERROR SUMMARY")
print("Mean error (km/h):", round(np.mean(errors), 3))
print("Mean absolute error (km/h):",
      round(np.mean(np.abs(errors)), 3))
print("Error standard deviation (km/h):",
      round(np.std(errors), 3))